# Dysarthria / ALS Speech Detection — HuBERT + LoRA (Fixed)
## TORGO Dataset · Colab T4 Safe · All Known Issues Resolved

### Fixes applied
| Issue | Fix |
|---|---|
| `transformers` requires PyTorch >= 2.4 | Pinned `transformers==4.40.2` + `peft==0.10.0` |
| `get_peft_model` fails on audio models | `inject_adapter_in_model` instead |
| DataLoader multiprocessing crash (Python 3.12) | `num_workers=0` |
| Feature extractor dtype inference error | Explicit `np.float32` cast + `truncation=True` |
| Random val split data leakage | Speaker-independent val & test splits |
| Frozen backbone → 40% accuracy | LoRA adapts `q_proj`/`v_proj` in every layer |


In [ ]:
# Do NOT reinstall torch — Colab's pre-installed version is correct
# Only install missing packages with pinned compatible versions
!pip install -q transformers==4.40.2 peft==0.10.0 accelerate librosa soundfile scikit-learn matplotlib seaborn
print("Install complete. Now do Runtime → Restart session, then run from Cell 2.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 30.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
Install complete. Now do Runtime → Restart session, then run from Cell 2.


In [ ]:
import os, re, gc, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

from transformers import HubertModel, Wav2Vec2FeatureExtractor
from peft import LoraConfig, inject_adapter_in_model
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score, precision_recall_curve)
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch : 2.10.0+cu128
Device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB


In [ ]:
from sklearn.metrics import classification_report
from torch.cuda.amp import autocast, GradScaler

In [ ]:
!kaggle datasets download -d pranaykoppula/torgo-audio -q
!unzip -o torgo-audio.zip -d /content/torgo > /dev/null
print("Dataset ready.")


Dataset URL: https://www.kaggle.com/datasets/pranaykoppula/torgo-audio
License(s): other
Dataset ready.


In [ ]:
def load_data(dataset_path="/content/torgo"):
    file_paths, labels = [], []
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if not file.endswith(".wav"):
                continue
            path = os.path.join(root, file)
            rl = root.lower()
            if "dys" in rl:
                labels.append(1)
            elif "con" in rl:
                labels.append(0)
            else:
                continue
            file_paths.append(path)
    return file_paths, np.array(labels)

file_paths, labels = load_data("/content/torgo")
if len(file_paths) == 0:
    file_paths, labels = load_data("/content")

print(f"Total      : {len(file_paths)}")
print(f"Control (0): {np.sum(labels==0)}")
print(f"Dysarth (1): {np.sum(labels==1)}")


Total      : 17635
Control (0): 11456
Dysarth (1): 6179


In [ ]:
# Test  : M01 (severe), M05 (mild), FC01, MC04
# Val   : F01 (severe), FC02   ← no speaker leakage into train
# Train : everyone else

TEST_SPEAKERS = {"M01", "M05", "FC01", "MC04"}
VAL_SPEAKERS  = {"F01", "FC02"}

def get_speaker(path):
    folder = os.path.basename(os.path.dirname(path))
    m = re.search(r"(MC\d+|FC\d+|M\d+|F\d+)", folder)
    return m.group(1) if m else "UNK"

X_train, y_train = [], []
X_val,   y_val   = [], []
X_test,  y_test  = [], []

for p, l in zip(file_paths, labels):
    s = get_speaker(p)
    if   s in TEST_SPEAKERS: X_test.append(p);  y_test.append(l)
    elif s in VAL_SPEAKERS:  X_val.append(p);   y_val.append(l)
    else:                    X_train.append(p); y_train.append(l)

y_train, y_val, y_test = map(np.array, [y_train, y_val, y_test])

for split, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{split:5s}: {len(y):5d}  ctrl={np.sum(y==0)}  dys={np.sum(y==1)}")


Train: 11588  ctrl=7100  dys=4488
Val  :  2529  ctrl=2261  dys=268
Test :  3518  ctrl=2095  dys=1423


In [ ]:
SR = 16000
MAX_SAMPLES = SR * 4

def preprocess_audio(path, augment=False):
    try:
        audio, _ = librosa.load(path, sr=SR, mono=True)
        if len(audio) < 400:
            return None
        audio = audio[:MAX_SAMPLES]
        audio = audio / (np.max(np.abs(audio)) + 1e-8)
        if augment and np.random.rand() > 0.5:
            rate = np.random.uniform(0.85, 1.15)
            audio = librosa.effects.time_stretch(audio, rate=rate)[:MAX_SAMPLES]
        if augment and np.random.rand() > 0.5:
            steps = np.random.randint(-2, 3)
            audio = librosa.effects.pitch_shift(audio, sr=SR, n_steps=steps)
        if np.isnan(audio).any():
            return None
        return audio.astype(np.float32)
    except Exception:
        return None



In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")

hubert = HubertModel.from_pretrained("facebook/hubert-base-ls960")
for name, param in hubert.named_parameters():
    if "encoder.layers" in name:
        layer_num = int(name.split("encoder.layers.")[1].split(".")[0])
        if layer_num < 6:
            param.requires_grad = False
hubert.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=None,
)

hubert_lora = inject_adapter_in_model(lora_config, hubert)

# Freeze everything; unfreeze only LoRA adapter params
for name, param in hubert_lora.named_parameters():
    param.requires_grad = "lora_" in name

total     = sum(p.numel() for p in hubert_lora.parameters())
trainable = sum(p.numel() for p in hubert_lora.parameters() if p.requires_grad)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  ({100*trainable/total:.2f}%)")

hubert_lora = hubert_lora.to(device)


preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/hubert-base-ls960 were not used when initializing HubertModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing HubertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing HubertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of HubertModel were not initialized from the model checkpoint at facebook/hubert-base-ls960 and are newly initialized: ['encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for pre

Total params     : 94,961,536
Trainable params : 589,824  (0.62%)


In [ ]:
class MeanStdPooling(nn.Module):
    def forward(self, x):
        mean = x.mean(dim=1)
        std  = x.std(dim=1).clamp(min=1e-6)
        return torch.cat([mean, std], dim=1)


class DysarthriaClassifier(nn.Module):
    def __init__(self, backbone, hidden=768, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.pool     = MeanStdPooling()
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Linear(hidden * 2, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )
    def forward(self, input_values, attention_mask=None):
        out    = self.backbone(input_values=input_values,
                               attention_mask=attention_mask)
        hidden = out.last_hidden_state
        pooled = self.pool(hidden)
        return self.head(pooled)


model = DysarthriaClassifier(hubert_lora).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total model params     : {total:,}")
print(f"Trainable model params : {trainable:,}  ({100*trainable/total:.2f}%)")


Total model params     : 95,374,593
Trainable model params : 1,002,881  (1.05%)


## Data Loading & Balanced Validation Split

In [ ]:
import random
from torch.utils.data import Subset

class TorgoDataset(Dataset):
    def __init__(self, paths, labels, augment):
        self.paths  = paths
        self.labels = labels
        self.augment=augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        audio = preprocess_audio(self.paths[idx], augment=self.augment)
        return audio, float(self.labels[idx])

dataset_train = TorgoDataset(X_train, y_train, augment=True)


def collate_fn(batch):
    batch = [(a, l) for a, l in batch if a is not None]
    if not batch:
        return None, None, None
    audios, labels = zip(*batch)
    audios = [np.array(a, dtype=np.float32) for a in audios]
    enc = feature_extractor(
        audios,
        sampling_rate=SR,
        return_tensors="pt",
        padding=True,
        return_attention_mask=True,
        max_length=MAX_SAMPLES,
        truncation=True,
    )
    return (enc["input_values"],
            enc["attention_mask"],
            torch.tensor(labels, dtype=torch.float32))


BATCH_SIZE = 4

# datasets
dataset_train = TorgoDataset(X_train, y_train, augment=True)
dataset_val   = TorgoDataset(X_val,   y_val, augment=False)   # full val set

# Balance validation: downsample majority class with fixed seed
random.seed(42)
ctrl_idx = [i for i, l in enumerate(y_val) if l == 0]
dys_idx  = [i for i, l in enumerate(y_val) if l == 1]
n_min    = min(len(ctrl_idx), len(dys_idx))
ctrl_sampled = random.sample(ctrl_idx, n_min)
dys_sampled  = random.sample(dys_idx,  n_min)
balanced_idx = ctrl_sampled + dys_sampled
random.shuffle(balanced_idx)

val_dataset_balanced = Subset(dataset_val, balanced_idx)

print(f"Full val     : ctrl={len(ctrl_idx)}  dys={len(dys_idx)}")
print(f"Balanced val : ctrl={n_min}  dys={n_min}  total={len(val_dataset_balanced)}")

#Weighted sampling
class_counts = np.bincount(y_train)
weights = 1.0 / class_counts[y_train]
sampler = WeightedRandomSampler(weights, len(weights))

train_loader = DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset_balanced,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    TorgoDataset(X_test, y_test, augment=False),
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False,
)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")


Full val     : ctrl=2261  dys=268
Balanced val : ctrl=268  dys=268  total=536

Train batches : 2897
Val   batches : 134
Test  batches : 440


In [ ]:
EPOCHS        = 6
learning_rate = 5e-5
PATIENCE = 3
patience_counter = 0

n_ctrl = np.sum(y_train == 0)
n_dys  = np.sum(y_train == 1)
pos_weight = torch.tensor([n_ctrl / n_dys], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)

from transformers import get_cosine_schedule_with_warmup
total_steps  = EPOCHS * len(train_loader)
warmup_steps = int(0.1 * total_steps)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
scaler    = GradScaler()

train_losses, val_losses = [], []
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_steps = 0.0, 0

    for iv, am, lbl in train_loader:
        if iv is None:
            continue
        iv  = iv.to(device)
        am  = am.to(device)
        lbl = lbl.unsqueeze(1).to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(iv, am)
            loss   = criterion(logits, lbl)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        t_loss  += loss.item()
        t_steps += 1

    avg_train = t_loss / max(t_steps, 1)
    train_losses.append(avg_train)          # FIX 4: record

    #Validate
    model.eval()
    v_loss, v_steps = 0.0, 0
    all_probs, all_true = [], []

    with torch.no_grad():
        for iv, am, lbl in val_loader:
            if iv is None:
                continue
            iv  = iv.to(device)
            am  = am.to(device)
            lbl = lbl.unsqueeze(1).to(device)
            with autocast():
                logits = model(iv, am)
                v_loss += criterion(logits, lbl).item()
            v_steps += 1
            all_probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            all_true.extend(lbl.cpu().numpy().flatten())

    avg_val = v_loss / max(v_steps, 1)
    val_losses.append(avg_val)              # FIX 4: record

    # FIX 5: use fixed 0.5 threshold for epoch-level monitoring ONLY
    #         final optimal threshold is found once post-training via PR curve
    preds     = (np.array(all_probs) > 0.5).astype(int)
    val_acc   = accuracy_score(all_true, preds)
    val_f1    = f1_score(all_true, preds, zero_division=0)

    print(f"Epoch {epoch+1:2d}/{EPOCHS}  "
          f"train={avg_train:.4f}  val={avg_val:.4f}  "
          f"acc={val_acc:.3f}  f1={val_f1:.3f}")

    if val_f1 > best_f1:
      best_f1 = val_f1
      patience_counter = 0
      torch.save(hubert_lora.state_dict(), "best_lora_adapter.pt")
      torch.save(model.head.state_dict(),  "best_head.pt")
      print(f"  ✓ Saved best checkpoint  (f1={best_f1:.4f})")
    else:
      patience_counter += 1
      print(f"  No improvement ({patience_counter}/{PATIENCE})")
      if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

    torch.cuda.empty_cache()
    gc.collect()

print("\nTraining complete.  Best val F1 (thresh=0.5) :", round(best_f1, 4))


Epoch  1/6  train=0.6794  val=0.2483  acc=0.925  f1=0.929
  ✓ Saved best checkpoint  (f1=0.9291)
Epoch  2/6  train=0.4320  val=0.6184  acc=0.858  f1=0.875
  No improvement (1/3)
Epoch  3/6  train=0.3405  val=0.3236  acc=0.923  f1=0.928
  No improvement (2/3)
Epoch  4/6  train=0.2890  val=0.4733  acc=0.904  f1=0.913
  No improvement (3/3)
Early stopping triggered.

Training complete.  Best val F1 (thresh=0.5) : 0.9291


In [ ]:
hubert_lora.load_state_dict(torch.load("best_lora_adapter.pt"))
model.head.load_state_dict(torch.load("best_head.pt"))
model.eval()

test_probs, test_true = [], []

with torch.no_grad():
    for iv, am, lbl in test_loader:
        if iv is None:
            continue

        with autocast():
            logits = model(iv.to(device), am.to(device))

        probs = torch.sigmoid(logits).cpu().numpy().flatten()

        test_probs.extend(probs)
        test_true.extend(lbl.numpy().flatten())


from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(test_true, test_probs)

# Pick lowest threshold where dysarthric precision >= 0.70
valid = np.where(precision[:-1] >= 0.70)[0]
best_thresh = thresholds[valid[0]] if len(valid) > 0 else 0.5

test_preds = (np.array(test_probs) > best_thresh).astype(int)
test_true  = np.array(test_true)

from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(test_true, test_preds))
print("\nClassification Report:\n")
print(classification_report(test_true, test_preds,
                            target_names=["Control", "Dysarthric"]))

Accuracy: 0.8169859879897055

Classification Report:

              precision    recall  f1-score   support

     Control       0.93      0.75      0.83      2094
  Dysarthric       0.71      0.91      0.80      1403

    accuracy                           0.82      3497
   macro avg       0.82      0.83      0.82      3497
weighted avg       0.84      0.82      0.82      3497

